# Fatal Police Shootings in the United States (2015-2017)
## Comprehensive Data Analysis with Socioeconomic Correlations

**Data Source:** The Washington Post Fatal Force Database  
**Analysis Period:** January 2015 - July 2017  
**Dataset:** Fatal police shootings combined with US Census socioeconomic data

---

## Table of Contents
1. [Setup & Data Loading](#setup)
2. [Temporal Analysis](#temporal)
3. [Demographic Analysis](#demographic)
4. [Incident Characteristics](#incident)
5. [Geographic Distribution](#geographic)
6. [Socioeconomic Analysis](#socioeconomic)
7. [Race & Disparity Analysis](#race)
8. [Key Findings](#findings)

---

<a id='setup'></a>
## 1. Setup & Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
import re
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Load datasets
deaths = pd.read_csv('Deaths_by_Police_US.csv', encoding='latin1')
income = pd.read_csv('Median_Household_Income_2015.csv', encoding='latin1')
hs_grad = pd.read_csv('Pct_Over_25_Completed_High_School.csv', encoding='latin1')
poverty = pd.read_csv('Pct_People_Below_Poverty_Level.csv', encoding='latin1')
race_demo = pd.read_csv('Share_of_Race_By_City.csv', encoding='latin1')

print("Datasets loaded successfully!")
print(f"Deaths dataset: {deaths.shape}")
print(f"Income dataset: {income.shape}")
print(f"HS Graduation dataset: {hs_grad.shape}")
print(f"Poverty dataset: {poverty.shape}")
print(f"Race Demographics dataset: {race_demo.shape}")

In [ ]:
# Preview deaths dataset
print("Deaths by Police Dataset:")
print("=" * 80)
deaths.head()

In [ ]:
# Dataset information
print("Column Information:")
deaths.info()

In [ ]:
# Check for missing values
print("Missing Values Analysis:")
missing = deaths.isnull().sum()
missing_pct = (missing / len(deaths) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

<a id='temporal'></a>
## 2. Temporal Analysis

In [ ]:
# Convert date column
deaths['date'] = pd.to_datetime(deaths['date'], format='%d/%m/%y')
deaths['year'] = deaths['date'].dt.year
deaths['month'] = deaths['date'].dt.month
deaths['month_name'] = deaths['date'].dt.month_name()

print("Temporal Data Summary:")
print(f"Date range: {deaths['date'].min()} to {deaths['date'].max()}")
print(f"Total shootings: {len(deaths):,}")
print(f"\nShootings by year:")
print(deaths['year'].value_counts().sort_index())

In [ ]:
# Monthly analysis
monthly_counts = deaths.groupby('month').size().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print("Monthly Statistics:")
for month, count in monthly_counts.items():
    print(f"{month_names[month-1]}: {count} shootings")

print(f"\nAverage per month: {monthly_counts.mean():.1f}")
print(f"Highest: {month_names[monthly_counts.idxmax()-1]} ({monthly_counts.max()})")
print(f"Lowest: {month_names[monthly_counts.idxmin()-1]} ({monthly_counts.min()})")

In [ ]:
# Visualize temporal trends
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Timeline plot
deaths.groupby(deaths['date'].dt.to_period('M')).size().plot(
    ax=axes[0], linewidth=2, color='#2E86AB', marker='o', markersize=4
)
axes[0].set_title('Fatal Police Shootings Over Time (2015-2017)', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Number of Shootings', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Monthly distribution
monthly_counts.plot(kind='bar', ax=axes[1], color='#F18F01')
axes[1].set_title('Fatal Shootings by Month (All Years Combined)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month', fontsize=12)
axes[1].set_ylabel('Total Count', fontsize=12)
axes[1].set_xticklabels(month_names, rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

<a id='demographic'></a>
## 3. Demographic Analysis

In [ ]:
# Race analysis
race_labels = {
    'W': 'White',
    'B': 'Black',
    'H': 'Hispanic',
    'A': 'Asian',
    'N': 'Native American',
    'O': 'Other'
}

print("RACE DISTRIBUTION")
print("="*80)
race_counts = deaths['race'].value_counts(dropna=False)
total_with_race = deaths['race'].notna().sum()

for race_code, count in race_counts.items():
    if pd.isna(race_code):
        print(f"Unknown: {count} ({count/len(deaths)*100:.1f}% of total)")
    else:
        race_name = race_labels.get(race_code, race_code)
        print(f"{race_name}: {count} ({count/total_with_race*100:.1f}% of known)")

print(f"\nTotal with known race: {total_with_race:,}")
print(f"Total with unknown race: {deaths['race'].isna().sum()}")

In [ ]:
# Gender analysis
print("\nGENDER DISTRIBUTION")
print("="*80)
gender_counts = deaths['gender'].value_counts()
for gender, count in gender_counts.items():
    gender_name = 'Male' if gender == 'M' else 'Female'
    print(f"{gender_name}: {count} ({count/len(deaths)*100:.1f}%)")

In [ ]:
# Age analysis
print("\nAGE STATISTICS")
print("="*80)
print(f"Mean age: {deaths['age'].mean():.1f} years")
print(f"Median age: {deaths['age'].median():.0f} years")
print(f"Standard deviation: {deaths['age'].std():.1f} years")
print(f"Age range: {deaths['age'].min():.0f} - {deaths['age'].max():.0f} years")
print(f"Missing values: {deaths['age'].isna().sum()} ({deaths['age'].isna().sum()/len(deaths)*100:.1f}%)")

print("\nAge distribution:")
print(deaths['age'].describe())

In [ ]:
# Age groups
age_bins = [0, 18, 25, 35, 45, 55, 65, 100]
age_labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
deaths['age_group'] = pd.cut(deaths['age'], bins=age_bins, labels=age_labels)

print("\nAGE GROUPS")
print("="*80)
age_group_counts = deaths['age_group'].value_counts().sort_index()
for group, count in age_group_counts.items():
    pct = count / deaths['age'].notna().sum() * 100
    print(f"{group}: {count} ({pct:.1f}%)")

In [ ]:
# Demographic visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Race pie chart
race_counts_known = deaths[deaths['race'].notna()]['race'].value_counts()
race_counts_known.index = [race_labels.get(x, x) for x in race_counts_known.index]
colors = plt.cm.Set3(range(len(race_counts_known)))
axes[0, 0].pie(race_counts_known, labels=race_counts_known.index, 
               autopct='%1.1f%%', startangle=90, colors=colors)
axes[0, 0].set_title('Victims by Race (Known)', fontsize=14, fontweight='bold')

# Gender pie chart
gender_counts.index = ['Male', 'Female']
axes[0, 1].pie(gender_counts, labels=gender_counts.index, 
               autopct='%1.1f%%', colors=['#3D5A80', '#EE6C4D'], startangle=90)
axes[0, 1].set_title('Victims by Gender', fontsize=14, fontweight='bold')

# Age distribution histogram
axes[1, 0].hist(deaths['age'].dropna(), bins=30, color='#06A77D', 
                edgecolor='black', alpha=0.7)
axes[1, 0].axvline(deaths['age'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f"Mean: {deaths['age'].mean():.1f}")
axes[1, 0].axvline(deaths['age'].median(), color='blue', linestyle='--', 
                   linewidth=2, label=f"Median: {deaths['age'].median():.0f}")
axes[1, 0].set_title('Age Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Age (years)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Age groups bar chart
age_group_counts.plot(kind='bar', ax=axes[1, 1], color='#A23B72')
axes[1, 1].set_title('Victims by Age Group', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Age Group')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

<a id='incident'></a>
## 4. Incident Characteristics

In [ ]:
# Manner of death
print("MANNER OF DEATH")
print("="*80)
manner_counts = deaths['manner_of_death'].value_counts()
for manner, count in manner_counts.items():
    print(f"{manner}: {count} ({count/len(deaths)*100:.1f}%)")

In [ ]:
# Armed status
print("\nARMED STATUS")
print("="*80)
armed_counts = deaths['armed'].value_counts().head(15)
total_armed = deaths['armed'].notna().sum()

for weapon, count in armed_counts.items():
    print(f"{weapon}: {count} ({count/total_armed*100:.1f}%)")

print(f"\nTotal categories: {deaths['armed'].nunique()}")
print(f"Missing values: {deaths['armed'].isna().sum()}")

In [ ]:
# Mental illness
print("\nSIGNS OF MENTAL ILLNESS")
print("="*80)
mental_counts = deaths['signs_of_mental_illness'].value_counts()
for status, count in mental_counts.items():
    status_label = 'Yes' if status else 'No'
    print(f"{status_label}: {count} ({count/len(deaths)*100:.1f}%)")

In [ ]:
# Threat level
print("\nTHREAT LEVEL")
print("="*80)
threat_counts = deaths['threat_level'].value_counts()
for level, count in threat_counts.items():
    print(f"{level}: {count} ({count/len(deaths)*100:.1f}%)")

In [ ]:
# Flee status
print("\nFLEE STATUS")
print("="*80)
flee_counts = deaths['flee'].value_counts(dropna=False)
total_flee = deaths['flee'].notna().sum()

for status, count in flee_counts.items():
    if pd.isna(status):
        print(f"Unknown: {count} ({count/len(deaths)*100:.1f}% of total)")
    else:
        print(f"{status}: {count} ({count/total_flee*100:.1f}% of known)")

In [ ]:
# Body camera
print("\nBODY CAMERA PRESENT")
print("="*80)
camera_counts = deaths['body_camera'].value_counts()
for status, count in camera_counts.items():
    status_label = 'Yes' if status else 'No'
    print(f"{status_label}: {count} ({count/len(deaths)*100:.1f}%)")

In [ ]:
# Incident characteristics visualizations
fig, axes = plt.subplots(3, 2, figsize=(16, 16))

# Armed status (top 10)
armed_top10 = deaths['armed'].value_counts().head(10)
armed_top10.plot(kind='barh', ax=axes[0, 0], color='#F18F01')
axes[0, 0].set_title('Top 10 Weapon Types', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Count')
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Mental illness pie
mental_counts.index = ['No Signs', 'Signs Present']
axes[0, 1].pie(mental_counts, labels=mental_counts.index, autopct='%1.1f%%',
               colors=['#6A994E', '#BC4749'], startangle=90)
axes[0, 1].set_title('Signs of Mental Illness', fontsize=14, fontweight='bold')

# Threat level
threat_counts.plot(kind='bar', ax=axes[1, 0], color='#C1121F')
axes[1, 0].set_title('Threat Level', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Threat Level')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Flee status
flee_counts_known = deaths['flee'].value_counts()
flee_counts_known.plot(kind='bar', ax=axes[1, 1], color='#669BBC')
axes[1, 1].set_title('Flee Status', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Flee Type')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Body camera pie
camera_counts.index = ['No Body Cam', 'Body Cam']
axes[2, 0].pie(camera_counts, labels=camera_counts.index, autopct='%1.1f%%',
               colors=['#EE6C4D', '#3D5A80'], startangle=90)
axes[2, 0].set_title('Body Camera Present', fontsize=14, fontweight='bold')

# Manner of death
manner_counts.plot(kind='bar', ax=axes[2, 1], color='#2A9D8F')
axes[2, 1].set_title('Manner of Death', fontsize=14, fontweight='bold')
axes[2, 1].set_xlabel('Method')
axes[2, 1].set_ylabel('Count')
axes[2, 1].tick_params(axis='x', rotation=45)
axes[2, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

<a id='geographic'></a>
## 5. Geographic Distribution

In [ ]:
# State analysis
print("TOP 20 STATES BY FATAL SHOOTINGS")
print("="*80)
state_counts = deaths['state'].value_counts().head(20)
for state, count in state_counts.items():
    print(f"{state}: {count} ({count/len(deaths)*100:.1f}%)")

print(f"\nTotal states: {deaths['state'].nunique()}")

In [ ]:
# City analysis
print("\nTOP 25 CITIES BY FATAL SHOOTINGS")
print("="*80)
city_state = deaths.groupby(['city', 'state']).size().sort_values(ascending=False).head(25)
for (city, state), count in city_state.items():
    print(f"{city}, {state}: {count}")

In [ ]:
# Geographic visualizations
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Top 20 states
state_counts.plot(kind='barh', ax=axes[0], color='#023047')
axes[0].set_title('Top 20 States by Fatal Shootings', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('State')
axes[0].grid(True, alpha=0.3, axis='x')

# Top 20 cities
city_state_top20 = deaths.groupby(['city', 'state']).size().sort_values(ascending=False).head(20)
city_labels = [f"{city}, {state}" for city, state in city_state_top20.index]
axes[1].barh(range(len(city_state_top20)), city_state_top20.values, color='#D62828')
axes[1].set_yticks(range(len(city_labels)))
axes[1].set_yticklabels(city_labels)
axes[1].set_title('Top 20 Cities by Fatal Shootings', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

<a id='socioeconomic'></a>
## 6. Socioeconomic Analysis

In [ ]:
# Function to clean city names for merging
def clean_city_name(city):
    """Remove city type suffixes (city, town, village, CDP, etc.)"""
    if pd.isna(city):
        return city
    city = re.sub(r'\s+(city|town|village|CDP|borough|municipality)$', 
                  '', str(city), flags=re.IGNORECASE)
    return city.strip().upper()

# Prepare deaths data
deaths['city_clean'] = deaths['city'].apply(clean_city_name)
deaths['state_clean'] = deaths['state'].str.upper()
deaths['city_state'] = deaths['city_clean'] + ', ' + deaths['state_clean']

# Prepare socioeconomic data
income['city_clean'] = income['City'].apply(clean_city_name)
income['state_clean'] = income['Geographic Area'].str.upper()
income['city_state'] = income['city_clean'] + ', ' + income['state_clean']
income['Median Income'] = pd.to_numeric(income['Median Income'], errors='coerce')

hs_grad['city_clean'] = hs_grad['City'].apply(clean_city_name)
hs_grad['state_clean'] = hs_grad['Geographic Area'].str.upper()
hs_grad['city_state'] = hs_grad['city_clean'] + ', ' + hs_grad['state_clean']
hs_grad['percent_completed_hs'] = pd.to_numeric(hs_grad['percent_completed_hs'], errors='coerce')

poverty['city_clean'] = poverty['City'].apply(clean_city_name)
poverty['state_clean'] = poverty['Geographic Area'].str.upper()
poverty['city_state'] = poverty['city_clean'] + ', ' + poverty['state_clean']
poverty['poverty_rate'] = pd.to_numeric(poverty['poverty_rate'], errors='coerce')

race_demo['city_clean'] = race_demo['City'].apply(clean_city_name)
race_demo['state_clean'] = race_demo['Geographic area'].str.upper()
race_demo['city_state'] = race_demo['city_clean'] + ', ' + race_demo['state_clean']
for col in ['share_white', 'share_black', 'share_hispanic', 'share_asian', 'share_native_american']:
    race_demo[col] = pd.to_numeric(race_demo[col], errors='coerce')

print("City names cleaned and standardized for merging")

In [ ]:
# Merge datasets
merged = deaths.copy()
merged = merged.merge(income[['city_state', 'Median Income']], on='city_state', how='left')
merged = merged.merge(hs_grad[['city_state', 'percent_completed_hs']], on='city_state', how='left')
merged = merged.merge(poverty[['city_state', 'poverty_rate']], on='city_state', how='left')
merged = merged.merge(race_demo[['city_state', 'share_white', 'share_black', 
                                   'share_native_american', 'share_asian', 'share_hispanic']], 
                      on='city_state', how='left')

print("MERGE SUCCESS RATES")
print("="*80)
print(f"Median Income: {merged['Median Income'].notna().sum()} / {len(merged)} "
      f"({merged['Median Income'].notna().sum()/len(merged)*100:.1f}%)")
print(f"HS Graduation: {merged['percent_completed_hs'].notna().sum()} / {len(merged)} "
      f"({merged['percent_completed_hs'].notna().sum()/len(merged)*100:.1f}%)")
print(f"Poverty Rate: {merged['poverty_rate'].notna().sum()} / {len(merged)} "
      f"({merged['poverty_rate'].notna().sum()/len(merged)*100:.1f}%)")
print(f"Race Demographics: {merged['share_white'].notna().sum()} / {len(merged)} "
      f"({merged['share_white'].notna().sum()/len(merged)*100:.1f}%)")

In [ ]:
# Socioeconomic statistics
print("\nSOCIOECONOMIC STATISTICS (Matched Cities)")
print("="*80)

print("\nMedian Household Income:")
print(f"  Mean: ${merged['Median Income'].mean():,.0f}")
print(f"  Median: ${merged['Median Income'].median():,.0f}")
print(f"  Std Dev: ${merged['Median Income'].std():,.0f}")
print(f"  Range: ${merged['Median Income'].min():,.0f} - ${merged['Median Income'].max():,.0f}")

print("\nHigh School Graduation Rate:")
print(f"  Mean: {merged['percent_completed_hs'].mean():.1f}%")
print(f"  Median: {merged['percent_completed_hs'].median():.1f}%")
print(f"  Std Dev: {merged['percent_completed_hs'].std():.1f}%")
print(f"  Range: {merged['percent_completed_hs'].min():.1f}% - {merged['percent_completed_hs'].max():.1f}%")

print("\nPoverty Rate:")
print(f"  Mean: {merged['poverty_rate'].mean():.1f}%")
print(f"  Median: {merged['poverty_rate'].median():.1f}%")
print(f"  Std Dev: {merged['poverty_rate'].std():.1f}%")
print(f"  Range: {merged['poverty_rate'].min():.1f}% - {merged['poverty_rate'].max():.1f}%")

print("\nRacial Demographics (Average):")
print(f"  White: {merged['share_white'].mean():.1f}%")
print(f"  Black: {merged['share_black'].mean():.1f}%")
print(f"  Hispanic: {merged['share_hispanic'].mean():.1f}%")
print(f"  Asian: {merged['share_asian'].mean():.1f}%")
print(f"  Native American: {merged['share_native_american'].mean():.1f}%")

In [ ]:
# Correlation analysis
print("\nCORRELATION MATRIX")
print("="*80)

corr_cols = ['Median Income', 'poverty_rate', 'percent_completed_hs', 
             'share_white', 'share_black', 'share_hispanic']
corr_data = merged[corr_cols].dropna()

print(f"Cases with complete data: {len(corr_data)} / {len(merged)} "
      f"({len(corr_data)/len(merged)*100:.1f}%)\n")

correlation_matrix = corr_data.corr()
print(correlation_matrix.round(3))

In [ ]:
# Socioeconomic visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Income distribution
merged['Median Income'].dropna().hist(bins=50, ax=axes[0, 0], 
                                       color='#06A77D', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(merged['Median Income'].mean(), color='red', 
                   linestyle='--', linewidth=2, label='Mean')
axes[0, 0].set_title('Distribution of City Median Income', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Median Income ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Poverty distribution
merged['poverty_rate'].dropna().hist(bins=50, ax=axes[0, 1], 
                                     color='#D62828', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(merged['poverty_rate'].mean(), color='blue', 
                   linestyle='--', linewidth=2, label='Mean')
axes[0, 1].set_title('Distribution of City Poverty Rate', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Poverty Rate (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# HS graduation distribution
merged['percent_completed_hs'].dropna().hist(bins=50, ax=axes[0, 2], 
                                             color='#2A9D8F', edgecolor='black', alpha=0.7)
axes[0, 2].axvline(merged['percent_completed_hs'].mean(), color='red', 
                   linestyle='--', linewidth=2, label='Mean')
axes[0, 2].set_title('Distribution of HS Graduation Rate', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('HS Graduation Rate (%)')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].legend()

# Income vs Poverty scatter
scatter_data = merged[['Median Income', 'poverty_rate']].dropna()
axes[1, 0].scatter(scatter_data['Median Income'], scatter_data['poverty_rate'], 
                   alpha=0.3, s=20, color='#E76F51')
z = np.polyfit(scatter_data['Median Income'], scatter_data['poverty_rate'], 1)
p = np.poly1d(z)
axes[1, 0].plot(scatter_data['Median Income'], 
                p(scatter_data['Median Income']), "r--", linewidth=2)
axes[1, 0].set_title('Median Income vs Poverty Rate', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Median Income ($)')
axes[1, 0].set_ylabel('Poverty Rate (%)')
axes[1, 0].grid(True, alpha=0.3)

# HS Grad vs Poverty scatter
scatter_data2 = merged[['percent_completed_hs', 'poverty_rate']].dropna()
axes[1, 1].scatter(scatter_data2['percent_completed_hs'], 
                   scatter_data2['poverty_rate'], alpha=0.3, s=20, color='#264653')
z2 = np.polyfit(scatter_data2['percent_completed_hs'], scatter_data2['poverty_rate'], 1)
p2 = np.poly1d(z2)
axes[1, 1].plot(scatter_data2['percent_completed_hs'], 
                p2(scatter_data2['percent_completed_hs']), "r--", linewidth=2)
axes[1, 1].set_title('HS Graduation vs Poverty Rate', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('HS Graduation Rate (%)')
axes[1, 1].set_ylabel('Poverty Rate (%)')
axes[1, 1].grid(True, alpha=0.3)

# Correlation heatmap
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, ax=axes[1, 2], cbar_kws={'shrink': 0.8})
axes[1, 2].set_title('Correlation Heatmap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

<a id='race'></a>
## 7. Race & Disparity Analysis

In [ ]:
# Race vs US population comparison
us_population_2015 = {
    'White': 61.6,
    'Black': 12.3,
    'Hispanic': 17.6,
    'Asian': 5.6,
    'Native American': 0.9,
    'Other': 2.0
}

victim_race_pct = {}
race_counts_known = deaths[deaths['race'].notna()]['race'].value_counts()
total_known = race_counts_known.sum()

for code, label in race_labels.items():
    if code in race_counts_known.index:
        victim_race_pct[label] = (race_counts_known[code] / total_known) * 100

print("RACE: VICTIMS vs US POPULATION")
print("="*80)
print(f"{'Race':<20} {'US Pop %':<15} {'Victim %':<15} {'Difference':<15}")
print("-"*80)
for race in ['White', 'Black', 'Hispanic', 'Asian']:
    us_pct = us_population_2015.get(race, 0)
    victim_pct = victim_race_pct.get(race, 0)
    diff = victim_pct - us_pct
    print(f"{race:<20} {us_pct:<15.1f} {victim_pct:<15.1f} {diff:+.1f}")

In [ ]:
# Unarmed rate by race
print("\nUNARMED VICTIM RATES BY RACE")
print("="*80)

for code, label in race_labels.items():
    if code in ['W', 'B', 'H', 'A']:
        race_data = deaths[deaths['race'] == code]
        total = len(race_data)
        unarmed = (race_data['armed'] == 'unarmed').sum()
        pct = (unarmed / total) * 100 if total > 0 else 0
        print(f"{label}: {unarmed} / {total} ({pct:.1f}%)")

In [ ]:
# Mental illness by race
print("\nMENTAL ILLNESS SIGNS BY RACE")
print("="*80)

mental_by_race = deaths[deaths['race'].notna()].groupby('race')['signs_of_mental_illness'].apply(
    lambda x: (x.sum() / len(x) * 100)
).sort_values(ascending=False)

for code, pct in mental_by_race.items():
    label = race_labels.get(code, code)
    print(f"{label}: {pct:.1f}%")

In [ ]:
# Average age by race
print("\nAVERAGE AGE BY RACE")
print("="*80)

age_by_race = deaths[deaths['race'].notna()].groupby('race')['age'].mean().sort_values()

for code, avg_age in age_by_race.items():
    label = race_labels.get(code, code)
    print(f"{label}: {avg_age:.1f} years")

In [ ]:
# Socioeconomic context by victim race
print("\nCITY CHARACTERISTICS BY VICTIM RACE")
print("="*80)

for code, label in race_labels.items():
    if code in ['W', 'B', 'H', 'A']:
        race_data = merged[merged['race'] == code]
        
        avg_income = race_data['Median Income'].mean()
        avg_poverty = race_data['poverty_rate'].mean()
        avg_hs = race_data['percent_completed_hs'].mean()
        
        print(f"\n{label} Victims (n={len(race_data)}):")
        if not pd.isna(avg_income):
            print(f"  Avg city median income: ${avg_income:,.0f}")
            print(f"  Avg city poverty rate: {avg_poverty:.1f}%")
            print(f"  Avg city HS grad rate: {avg_hs:.1f}%")

In [ ]:
# Income quintile analysis
merged_complete = merged[merged['Median Income'].notna()].copy()
merged_complete['income_quintile'] = pd.qcut(
    merged_complete['Median Income'], 
    q=5, 
    labels=['Lowest', 'Low', 'Middle', 'High', 'Highest']
)

print("\nANALYSIS BY CITY INCOME QUINTILE")
print("="*80)

for quintile in ['Lowest', 'Low', 'Middle', 'High', 'Highest']:
    quintile_data = merged_complete[merged_complete['income_quintile'] == quintile]
    
    avg_poverty = quintile_data['poverty_rate'].mean()
    avg_hs = quintile_data['percent_completed_hs'].mean()
    unarmed_pct = (quintile_data['armed'] == 'unarmed').sum() / len(quintile_data) * 100
    mental_pct = quintile_data['signs_of_mental_illness'].sum() / len(quintile_data) * 100
    
    print(f"\n{quintile} Income Cities (n={len(quintile_data)}):")
    print(f"  Avg poverty rate: {avg_poverty:.1f}%")
    print(f"  Avg HS grad rate: {avg_hs:.1f}%")
    print(f"  Unarmed victims: {unarmed_pct:.1f}%")
    print(f"  Mental illness signs: {mental_pct:.1f}%")

In [ ]:
# Comprehensive race analysis visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Detailed Race Analysis: Victim Demographics and City Characteristics', 
             fontsize=16, fontweight='bold')

# 1. Victims vs Population
races = ['White', 'Black', 'Hispanic', 'Asian']
x = np.arange(len(races))
width = 0.35

pop_pcts = [us_population_2015[r] for r in races]
victim_pcts = [victim_race_pct.get(r, 0) for r in races]

axes[0, 0].bar(x - width/2, pop_pcts, width, label='US Population %', 
               color='#577590', alpha=0.7)
axes[0, 0].bar(x + width/2, victim_pcts, width, label='Victim %', 
               color='#F94144', alpha=0.7)
axes[0, 0].set_ylabel('Percentage (%)')
axes[0, 0].set_title('Victim Race vs US Population', fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(races, rotation=45, ha='right')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Unarmed rate by race
unarmed_rates = []
for code in ['W', 'B', 'H', 'A']:
    race_data = deaths[deaths['race'] == code]
    rate = (race_data['armed'] == 'unarmed').sum() / len(race_data) * 100
    unarmed_rates.append(rate)

bars = axes[0, 1].bar(races, unarmed_rates, color=['#8ECAE6', '#219EBC', '#023047', '#FFB703'])
axes[0, 1].set_ylabel('Percentage (%)')
axes[0, 1].set_title('Rate of Unarmed Victims by Race', fontweight='bold')
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{unarmed_rates[i]:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Mental illness by race
mental_rates = []
for code in ['W', 'B', 'H', 'A']:
    race_data = deaths[deaths['race'] == code]
    rate = race_data['signs_of_mental_illness'].sum() / len(race_data) * 100
    mental_rates.append(rate)

bars = axes[0, 2].bar(races, mental_rates, color=['#90BE6D', '#43AA8B', '#577590', '#F94144'])
axes[0, 2].set_ylabel('Percentage (%)')
axes[0, 2].set_title('Rate of Mental Illness Signs by Race', fontweight='bold')
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0, 2].text(bar.get_x() + bar.get_width()/2., height,
                    f'{mental_rates[i]:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Average age by race
avg_ages = []
for code in ['W', 'B', 'H', 'A']:
    avg_age = deaths[deaths['race'] == code]['age'].mean()
    avg_ages.append(avg_age)

bars = axes[1, 0].bar(races, avg_ages, color=['#E76F51', '#F4A261', '#E9C46A', '#2A9D8F'])
axes[1, 0].set_ylabel('Years')
axes[1, 0].set_title('Average Age of Victims by Race', fontweight='bold')
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{avg_ages[i]:.1f}', ha='center', va='bottom', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 5. City poverty by victim race
poverty_by_race = []
for code in ['W', 'B', 'H', 'A']:
    avg_pov = merged[merged['race'] == code]['poverty_rate'].mean()
    poverty_by_race.append(avg_pov)

bars = axes[1, 1].bar(races, poverty_by_race, color=['#D62828', '#F77F00', '#FCBF49', '#06A77D'])
axes[1, 1].set_ylabel('Poverty Rate (%)')
axes[1, 1].set_title('Avg City Poverty Rate\n(Where Victims Were Shot)', fontweight='bold')
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{poverty_by_race[i]:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 6. City income by victim race
income_by_race = []
for code in ['W', 'B', 'H', 'A']:
    avg_inc = merged[merged['race'] == code]['Median Income'].mean()
    income_by_race.append(avg_inc)

bars = axes[1, 2].bar(races, income_by_race, color=['#06A77D', '#118AB2', '#073B4C', '#EF476F'])
axes[1, 2].set_ylabel('Median Income ($)')
axes[1, 2].set_title('Avg City Median Income\n(Where Victims Were Shot)', fontweight='bold')
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1, 2].text(bar.get_x() + bar.get_width()/2., height,
                    f'${income_by_race[i]:,.0f}', ha='center', va='bottom', 
                    fontweight='bold', fontsize=9)
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

<a id='findings'></a>
## 8. Key Findings & Critical Observations

### Summary Statistics

In [ ]:
print("COMPREHENSIVE SUMMARY")
print("="*80)
print(f"Total fatal shootings: {len(deaths):,}")
print(f"Date range: {deaths['date'].min().strftime('%B %d, %Y')} to {deaths['date'].max().strftime('%B %d, %Y')}")
print(f"States covered: {deaths['state'].nunique()}")
print(f"Cities covered: {deaths['city'].nunique():,}")
print(f"Socioeconomic data match rate: {merged['Median Income'].notna().sum()/len(merged)*100:.1f}%")

print("\nKEY STATISTICS:")
print(f"  Male victims: {(deaths['gender']=='M').sum()/len(deaths)*100:.1f}%")
print(f"  Average age: {deaths['age'].mean():.1f} years")
print(f"  Armed with gun: {(deaths['armed']=='gun').sum()/deaths['armed'].notna().sum()*100:.1f}%")
print(f"  Unarmed: {(deaths['armed']=='unarmed').sum()/deaths['armed'].notna().sum()*100:.1f}%")
print(f"  Mental illness signs: {deaths['signs_of_mental_illness'].sum()/len(deaths)*100:.1f}%")
print(f"  Body camera present: {deaths['body_camera'].sum()/len(deaths)*100:.1f}%")

### Critical Observations

#### 1. Racial Disparities
- **Black Americans** are disproportionately represented among victims (26% of victims vs 12% of US population)
- **Black victims** are nearly **twice as likely** to be unarmed compared to White victims (10.2% vs 5.6%)
- **Black and Hispanic victims** are significantly younger on average (31.6 and 33.0 years) compared to White victims (40.0 years)

#### 2. Mental Health Crisis
- **1 in 4 victims** showed signs of mental illness
- White victims show the highest rate of mental illness indicators (31.8%)
- Suggests significant gaps in mental health crisis response systems

#### 3. Use of Force Context
- 55.3% of victims were armed with guns
- 6.8% were completely unarmed
- 4.1% were carrying toy weapons
- Only 10.7% of incidents involved body cameras

#### 4. Socioeconomic Patterns
- Strong negative correlation between income and poverty (r = -0.75)
- Moderate correlation between education and poverty (r = -0.51)
- Higher-income cities show higher rates of unarmed victims and mental illness indicators
- Shootings occur across all income levels, not concentrated in poorest areas

#### 5. Geographic Concentration
- California accounts for 16.7% of all fatal shootings
- Large urban areas (Los Angeles, Phoenix, Houston) have highest absolute numbers
- Top 3 states (CA, TX, FL) account for 31.7% of all incidents

#### 6. Accountability Infrastructure
- Only 10.7% of incidents had body camera footage
- Highlights limited accountability mechanisms during 2015-2017 period
- Significant room for improvement in documentation and transparency

### Data Quality & Limitations

- Race data missing for 7.7% of cases
- Age data missing for 3.0% of cases  
- Socioeconomic data successfully merged for 85.3% of cases
- Database relies on news reports and public records - may have undercount
- Only includes fatal shootings - non-fatal incidents not represented
- City-level socioeconomic data may not reflect specific neighborhoods where incidents occurred

---

## Project Reflection

**Write your project reflection here:**

### How I Approached This Project:


### What Was Hard:


### What Was Easy:


### My Biggest Learning:


### What I Would Do Differently:


### Key Takeaways:


---